In [1]:
# Install the latest Ultralytics library (Supports YOLO11)
!pip install -U ultralytics pandas opencv-python tqdm

import os
import shutil
import json
import cv2
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm
from IPython.display import FileLink

print("✅ Libraries Installed Successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.1
    Uninstalling tqdm-4.67.1:
      Successfully uninstalled tqdm-4.67.1
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 4.12.0.88
    Uninstalling opencv-python-4.12.0.88:
      Successfully uninstalled opencv-python-4.12.0.88
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:
      Successfully uninstalled pandas-2.3.3
ERROR: pip's dependency resolver do

In [2]:
# --- CONFIGURATION FOR TRASHCAN 1.0 ---
SOURCE_JSON = "/kaggle/input/trashcan-1-0/dataset/material_version/instances_train_trashcan.json"
SOURCE_IMAGES = "/kaggle/input/trashcan-1-0/dataset/material_version/train"

DEST_BASE = "/kaggle/working/yolo_dataset"
DEST_IMAGES = os.path.join(DEST_BASE, "images/train")
DEST_LABELS = os.path.join(DEST_BASE, "labels/train")

# Create folders
os.makedirs(DEST_IMAGES, exist_ok=True)
os.makedirs(DEST_LABELS, exist_ok=True)

print("📂 Loading JSON Annotations...")
with open(SOURCE_JSON) as f:
    data = json.load(f)

images_info = {img['id']: img for img in data['images']}

print("🔄 Converting to YOLO format & Copying Images (This takes a minute)...")
for ann in tqdm(data['annotations']):
    img_id = ann['image_id']
    img_data = images_info[img_id]
    file_name = img_data['file_name']
    
    # Calculate YOLO BBox
    img_w, img_h = img_data['width'], img_data['height']
    x, y, w, h = ann['bbox']
    x_center, y_center = (x + w / 2) / img_w, (y + h / 2) / img_h
    width, height = w / img_w, h / img_h
    cat_id = ann['category_id'] - 1 

    # Save Label File
    txt_name = os.path.splitext(file_name)[0] + ".txt"
    with open(os.path.join(DEST_LABELS, txt_name), 'a') as f:
        f.write(f"{cat_id} {x_center} {y_center} {width} {height}\n")
        
    # Copy Image File
    src_img = os.path.join(SOURCE_IMAGES, file_name)
    dst_img = os.path.join(DEST_IMAGES, file_name)
    if os.path.exists(src_img) and not os.path.exists(dst_img):
        shutil.copy(src_img, dst_img)

print("✅ Dataset Prepared!")

📂 Loading JSON Annotations...
🔄 Converting to YOLO format & Copying Images (This takes a minute)...


100%|██████████| 9741/9741 [00:51<00:00, 190.20it/s]

✅ Dataset Prepared!


In [3]:
# Apply CLAHE to enhance underwater visibility
clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
image_files = [f for f in os.listdir(DEST_IMAGES) if f.endswith(('.jpg', '.png'))]

print(f"🌊 Enhancing {len(image_files)} images for better detection...")

for filename in tqdm(image_files):
    img_path = os.path.join(DEST_IMAGES, filename)
    img = cv2.imread(img_path)
    if img is not None:
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        l_enhanced = clahe.apply(l)
        final_img = cv2.cvtColor(cv2.merge((l_enhanced, a, b)), cv2.COLOR_LAB2BGR)
        cv2.imwrite(img_path, final_img) 

print("✅ Enhancement Complete!")

🌊 Enhancing 6008 images for better detection...


100%|██████████| 6008/6008 [00:30<00:00, 198.96it/s]

✅ Enhancement Complete!


In [4]:
# Write the data.yaml file needed for training
yaml_content = """
path: /kaggle/working/yolo_dataset
train: images/train
val: images/train  

nc: 22
names:
  0: rov
  1: plant
  2: animal_fish
  3: animal_starfish
  4: animal_shells
  5: animal_crab
  6: animal_eel
  7: animal_etc
  8: trash_clothing
  9: trash_pipe
  10: trash_bottle
  11: trash_bag
  12: trash_snack_wrapper
  13: trash_can
  14: trash_cup
  15: trash_container
  16: trash_unknown_instance
  17: trash_branch
  18: trash_wreckage
  19: trash_tarp
  20: trash_rope
  21: trash_net
"""
with open("/kaggle/working/data.yaml", "w") as f:
    f.write(yaml_content)
print("✅ data.yaml created!")

✅ data.yaml created!


In [5]:
# We use YOLO11s. It is the best supported model for this task.
MODEL_NAME = "yolo11s.pt" 

# --- CONFIGURATION ---
tuning_epochs = 5      # Quick tests
final_epochs = 50      # Long run for max confidence
project_dir = "/kaggle/working/runs"

ofat_stages = {
    "lr0": [0.01, 0.001],           
    "optimizer": ["auto", "AdamW"]  
}

best_params = {"lr0": 0.01, "optimizer": "auto", "batch": 16, "imgsz": 640}

print(f"🚀 Starting Advanced OFAT Tuning using {MODEL_NAME}...")

# --- PHASE 1: HYPERPARAMETER TUNING ---
for param, values in ofat_stages.items():
    stage_best_val = None
    stage_best_map = -1.0
    
    for val in values:
        print(f"\n🧪 Testing {param} = {val}")
        current_args = best_params.copy()
        current_args[param] = val
        
        # Download and Load Model
        model = YOLO(MODEL_NAME) 
        
        results = model.train(
            data="/kaggle/working/data.yaml",
            epochs=tuning_epochs,
            batch=current_args["batch"],
            lr0=current_args["lr0"],
            optimizer=current_args["optimizer"],
            project=project_dir,
            name=f"tune_{param}_{val}",
            plots=False,
            verbose=True # Logs ON so you see it working
        )
        
        current_map = results.results_dict.get('metrics/mAP50-95(B)', 0)
        print(f"📊 Result: mAP50-95 = {current_map:.4f}")
        
        if current_map > stage_best_map:
            stage_best_map = current_map
            stage_best_val = val
            
    best_params[param] = stage_best_val
    print(f"🏆 Best {param} found: {stage_best_val}")

print("\n" + "="*40)
print(f"🔥 STARTING FINAL MAX-CONFIDENCE TRAINING 🔥")
print(f"Optimized Parameters: {best_params}")
print("="*40)

# --- PHASE 2: FINAL TRAINING ---
final_model = YOLO(MODEL_NAME)
final_model.train(
    data="/kaggle/working/data.yaml",
    epochs=final_epochs,
    batch=best_params["batch"],
    lr0=best_params["lr0"],
    optimizer=best_params["optimizer"],
    project=project_dir,
    name="YOLO_MAX_CONF",
    mosaic=1.0,  # Strong augmentation
    mixup=0.15, 
    plots=True,
    save=True,
    verbose=True
)

print("\n🎉 TRAINING COMPLETE!")

🚀 Starting Advanced OFAT Tuning using yolo11s.pt...

🧪 Testing lr0 = 0.01
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=tune_lr0_0.01, nbs=64, nms=False, opset=None, 

In [6]:
from IPython.display import FileLink
import os

best_model_path = "/kaggle/working/runs/YOLO_MAX_CONF/weights/best.pt"

if os.path.exists(best_model_path):
    print("✅ Success! Your model is ready.")
    print("⬇️ Click the link below to download:")
    display(FileLink(best_model_path))
else:
    print("❌ Weights not found. Look in /kaggle/working/runs/YOLO_MAX_CONF/weights/")

✅ Success! Your model is ready.
⬇️ Click the link below to download:


/kaggle/working/runs/YOLO_MAX_CONF/weights/best.pt

In [7]:
import os
import shutil
from IPython.display import FileLink

# 1. Define the current location and the new location
source_path = "/kaggle/working/runs/YOLO_MAX_CONF/weights/best.pt"
new_path = "/kaggle/working/best_yolov12.pt"

# 2. Check if file exists and move it
if os.path.exists(source_path):
    shutil.copy(source_path, new_path)
    print("✅ Success! File moved to main directory.")
    print("⬇️ Click the link below to download:")
    # Linking to the file in the root directory usually fixes the 404 error
    display(FileLink("best_yolov12.pt"))
else:
    print("❌ Error: File still not found. Please check the Output sidebar manually.")

✅ Success! File moved to main directory.
⬇️ Click the link below to download:


/kaggle/working/best_yolov12.pt